# SmartHVAC Studio: Backend Worker (Layer 3)
This executable notebook acts as the **Coordination Engine**. It polls Firebase for new jobs, runs AI generation (Layer 4), executes EnergyPlus simulations (Layer 5), and uploads results.

---

In [5]:
from google.colab import drive
import os

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Change the working directory to your Drive's colab folder
project_path = "/content/drive/MyDrive/SmartHVAC-Studio/colab"
os.chdir(project_path)

print(f"Current Working Directory changed to: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current Working Directory changed to: /content/drive/MyDrive/SmartHVAC-Studio/colab


In [6]:
!python verify_phase2.py


Initializing AIPipelines...

User Prompt: I want a perfectly square house that is exactly 12 meters long and 12 meters wide. Make the height 3.5 meters. I want the exterior walls to use 2x4 Wood Stud R11 construction, and the roof also to use wood stud construction.

Running Generation with gemini...
[AI] Generating Modular IDF using model: gemini
[AI Assembler] AI Selected -> L:12.0, W:12.0, Wall:Composite 2x4 Wood Stud R11

--- FINAL ASSEMBLED IDF PREVIEW (first 1000 chars) ---

  Version,25.2;

  Building,
    Exercise 1A,             !- Name
    0.0,                     !- North Axis {deg}
    Country,                 !- Terrain
    0.04,                    !- Loads Convergence Tolerance Value {W}
    0.4,                     !- Temperature Convergence Tolerance Value {deltaC}
    FullInteriorAndExterior, !- Solar Distribution
    25,                      !- Maximum Number of Warmup Days
    6;                       !- Minimum Number of Warmup Days

  Timestep,4;

  SimulationContr

## 1. Setup Environment

In [7]:
# 1. Install Backend Dependencies
!pip install -r requirements.txt

# 2. Install Advisor's EnergyPlus Utility (from GitHub)
# Note: This installs the specific dev branch required for EKF hooks
!pip install -q "energy-plus-utility @ git+https://github.com/mugalan/energy-plus-utility.git@dev"

print("Dependencies installed.")

INFO: pip is looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 4.3 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies installed.


In [8]:
import sys
import os

# Add the local folder to path so we can import 'backend'
sys.path.append(os.getcwd())

print("Current Working Directory:", os.getcwd())
print("Python Path Updated.")

Current Working Directory: /content/drive/MyDrive/SmartHVAC-Studio/colab
Python Path Updated.


## 2. Authentication

In [9]:
# Check for Service Account Key
key_path = "serviceAccountKey.json"

if not os.path.exists(key_path):
    print("❌ ERROR: serviceAccountKey.json not found.")
    print("Please upload your Firebase Service Account JSON key to the file browser on the left.")
else:
    print("✅ Found serviceAccountKey.json")

✅ Found serviceAccountKey.json


## 3. Initialize Modules

In [10]:
from backend.firebase_connector import FirebaseConnector
from backend.ai_generator import AIPipelines
from simulation_runner import run_simulation_job
import time

# Initialize Connections
try:
    fb = FirebaseConnector(key_path)
    ai = AIPipelines() # API Key can be passed here or set in ENV
    print("Backend Modules Initialized Successfully.")
except Exception as e:
    print("Initialization Failed:", e)

[Firebase] Initializing with bucket: smarthvac-studio-b84c4.firebasestorage.app
[Firebase] Connected to project: smarthvac-studio-b84c4
Backend Modules Initialized Successfully.


In [11]:
import firebase_admin
from firebase_admin import credentials, firestore, storage
import json
import time
import difflib # For Diff Viewer

## 4. Verify epw found and downloaded to runtime

In [ ]:
import os

# Expected path relative to Colab root
file_path = "weather/USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw"

if os.path.exists(file_path):
    print(f"✅ FOUND: {file_path}")
    print(f"   Size: {os.path.getsize(file_path)/1024:.1f} KB")
else:
    print(f"❌ MISSING: {file_path}")
    if os.path.exists("weather"):
        print(f"   Contents of 'weather/': {os.listdir('weather')}")
    else:
        print("   'weather' folder does not exist!")


## 5. Verify Base.idf found and downloaded to runtime

In [ ]:
import os
import firebase_admin
from firebase_admin import storage

def download_template(template_name="Base.idf"):
    """Downloads the IDF template from Firebase Storage to local Colab."""
    try:
        bucket = storage.bucket()
        blob_path = f"templates/{template_name}"
        local_path = f"templates/{template_name}"

        # Ensure local dir exists
        os.makedirs("templates", exist_ok=True)

        blob = bucket.blob(blob_path)
        if blob.exists():
            blob.download_to_filename(local_path)
            print(f"✅ Downloaded Template: {local_path}")
            return True
        else:
            print(f"⚠️ Template not found in Storage: {blob_path}")
            print("   (Please create 'templates' folder in Firebase Storage and upload Base.idf)")
            return False

    except Exception as e:
        print(f"❌ Failed to download template: {e}")
        return False

        return False

def generate_and_upload_diff(base_path, new_path, job_id):
    """Generates an HTML diff between Base.idf and in.idf and uploads it."""
    try:
        # Read files
        with open(base_path, 'r') as f:
            base_lines = f.readlines()
        with open(new_path, 'r') as f:
            new_lines = f.readlines()

        # Generate HTML Diff using Python's built-in difflib
        diff_html = difflib.HtmlDiff().make_file(
            base_lines, new_lines,
            fromdesc='Base Template',
            todesc='Generated IDF',
            context=True,  # Show only changes + context
            numlines=3     # 3 lines of context
        )

        # Save locally
        diff_path = f"jobs/{job_id}/diff.html"
        with open(diff_path, "w") as f:
            f.write(diff_html)

        # Upload to Storage (Content-Type is CRITICAL for browser viewing)
        bucket = storage.bucket()
        blob = bucket.blob(f"jobs/{job_id}/diff.html")
        blob.upload_from_filename(diff_path, content_type="text/html")
        print(f"   📊 Uploaded Diff HTML to Storage.")
        return True
    except Exception as e:
        print(f"   ❌ Failed to generate diff: {e}")
        return False

# Call this on startup
download_template()


## 6. Main polling loop

In [ ]:
import time
from datetime import datetime
import traceback

def run_backend_loop(db):
    print("🚀 Backend is running... Waiting for jobs.")
    print("   - Watching 'jobs' collection for 'queued' tasks.")
    print("   - Watching 'test_connectivity' collection for 'test_connection' tasks.")

    while True:
        try:
            # ---------------------------------------------------------
            # 1. Check for CONNECTION TESTS (Fast Path)
            # ---------------------------------------------------------
            test_docs = db.collection("test_connectivity").where("status", "==", "test_connection").stream()

            for doc in test_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔹 Found Connection Test: {job_id}")

                # Get Toggles (Default to True if missing)
                do_gemini = data.get('checkGemini', True)
                do_openai = data.get('checkOpenAI', True)
                do_hf = data.get('checkHF', True)

                # Perform checks using your existing AI class
                # Pass the flags to the method
                results = ai.test_connections(check_openai=do_openai, check_gemini=do_gemini, check_hf=do_hf)

                # Update Firestore
                db.collection("test_connectivity").document(job_id).update({
                    "status": "tested",
                    "testResults": results,
                    "processedAt": datetime.now()
                })
                print(f"   ✅ Test Completed. Updated {job_id}")

            # ---------------------------------------------------------
            # 2. Check for SIMULATION JOBS (Normal Path)
            # ---------------------------------------------------------
            job_docs = db.collection("jobs").where("status", "==", "queued").stream()

            for doc in job_docs:
                job_id = doc.id
                data = doc.to_dict()
                print(f"🔸 Found Queued Job: {job_id}")

                # Mark as running immediately
                db.collection("jobs").document(job_id).update({"status": "running"})

                try:
                    # Extract Inputs from new UI
                    nlp_input = data.get("nlpInputText", "")
                    sim_settings = data.get("simulationConfig", {})

                    # Log what we received
                    print(f"   Input: {nlp_input[:50]}...")
                    print(f"   Settings: {sim_settings}")

                    # --- PHASE 3: THE SYSTEM LOOP ---
                    selected_model = data.get("selectedModel", "openai")
                    print(f"   [Phase 3] Generating IDF using {selected_model}...")

                    final_idf_string = ai.generate_idf_from_text(nlp_input, sim_settings, model_type=selected_model)

                    if final_idf_string.startswith("! Error"):
                        raise Exception(f"AI Generation Failed: {final_idf_string}")

                    # Save IDF
                    import os
                    idf_save_path = f"RunFiles/{job_id}_in.idf"
                    os.makedirs("RunFiles", exist_ok=True)
                    with open(idf_save_path, "w", encoding="utf-8") as f:
                        f.write(final_idf_string)

                    print("   [Phase 3] Running Simulation via simulation_runner.py...")
                    import simulation_runner

                    # Epw path usually defaults to "weather.epw" or from sim_settings
                    epw_name = sim_settings.get("weather_file", "weather.epw")
                    if not os.path.exists(epw_name) and os.path.exists("weather/" + epw_name):
                         epw_name = "weather/" + epw_name
                    elif not os.path.exists(epw_name):
                         epw_name = "weather.epw"

                    sim_results = simulation_runner.run_simulation_job(
                        job_id=job_id,
                        idf_path=idf_save_path,
                        epw_path=epw_name,
                        config=sim_settings,
                        output_dir_base="sim_runs"
                    )
                    print("   [Phase 3] Simulation Complete! Uploading results to Storage...")

                    # Upload to Storage
                    bucket = storage.bucket()
                    result_urls = {}

                    for key, file_path in sim_results.items():
                        if os.path.exists(file_path):
                            blob_path = f"jobs/{job_id}/{os.path.basename(file_path)}"
                            blob = bucket.blob(blob_path)

                            c_type = "image/png" if key == "plot" else "text/csv"
                            blob.upload_from_filename(file_path, content_type=c_type)

                            # Store the direct URL
                            try:
                                blob.make_public()
                                result_urls[key] = blob.public_url
                            except:
                                result_urls[key] = f"gs://{bucket.name}/{blob_path}"

                    db.collection("jobs").document(job_id).update({
                        "status": "done",
                        "resultPaths": result_urls,
                        "completedAt": datetime.now()
                    })
                    print(f"   ✅ Job Finished: {job_id}\n")

                except Exception as e:
                    error_msg = str(e)
                    print(f"   ❌ Job Failed: {error_msg}")
                    traceback.print_exc()

                    db.collection("jobs").document(job_id).update({
                        "status": "error",
                        "errorMessage": error_msg
                    })

            # Sleep to prevent quota exhaustion
            time.sleep(3)

        except Exception as main_e:
            print(f"⚠️ Loop Error: {main_e}")
            time.sleep(5)

# ---------------------------------------------------------------------
# START THE LOOP (Runs indefinitely)
# Ensure 'fb' is defined (from previous cells)
# ---------------------------------------------------------------------
if __name__ == "__main__":
    if 'fb' in globals():
        # Access the internal firestore client from your connector
        if hasattr(fb, 'db'):
            db_client = fb.db
        else:
            # Fallback if fb is the client itself
            db_client = fb

        run_backend_loop(db_client)
    else:
        print("❌ Error: 'fb' variable not found. Please run the Firebase Initialization cell first.")


🚀 Backend is running... Waiting for jobs.
   - Watching 'jobs' collection for 'queued' tasks.
   - Watching 'test_connectivity' collection for 'test_connection' tasks.


/usr/local/lib/python3.12/dist-packages/google/cloud/firestore_v1/base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


🔹 Found Connection Test: test_ai_20260419_005101
[AI] HuggingFace Test: ✅ Success
   ✅ Test Completed. Updated test_ai_20260419_005101
🔸 Found Queued Job: job_20260419_005128
   Input: Build an 18 by 18 meter office, 4 meters tall, usi...
   Settings: {'weather_file': 'USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw', 'run_type': 'design_day'}
   [Phase 3] Generating IDF using huggingface...
[AI] Generating Modular IDF using model: huggingface
[AI Assembler] AI Selected -> L:18.0, W:18.0, Wall:Composite Brick Foam 2x4 Steel Stud R11
   [Phase 3] Running Simulation via simulation_runner.py...
[job_20260419_005128] Bootstrapping EnergyPlus (Verbose)...
